# Peptide mass walkthrough with `peptide-mass`

By Peptide.Express (https://peptide.express)

This notebook installs the open-source `peptide-mass` package, loads the reference table from this repository, computes monoisotopic and average masses plus [M+nH]n+ values, and checks them against `data/masses.csv`.

## 1. Install

In [1]:
%pip install -q peptide-mass pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Load the reference table

In [2]:
import pandas as pd
BASE = 'https://raw.githubusercontent.com/Peptide-Express/peptide-reference-data/main/data/'
peps = pd.read_csv(BASE + 'peptides.csv')
ref = pd.read_csv(BASE + 'masses.csv')
print(len(peps), 'peptide rows;', len(ref), 'mass rows')
assert len(peps) == len(ref)

23 peptide rows; 23 mass rows


In [3]:
# Keep a short, neutral set of example rows for display
EXAMPLES = ['GHK', 'Epitalon', 'KPV', 'DSIP', 'Kisspeptin-10', 'Oxytocin', 'Gonadorelin']
ex = peps[peps['name'].isin(EXAMPLES)].reset_index(drop=True)
ex[['name', 'sequence', 'n_acetyl', 'c_amide', 'pyroglu', 'disulfides']]

,name,sequence,n_acetyl,c_amide,pyroglu,disulfides
0,GHK,GHK,0,0,0,0
1,Epitalon,AEDG,0,0,0,0
2,KPV,KPV,0,0,0,0
3,DSIP,WAGGDASGE,0,0,0,0
4,Kisspeptin-10,YNWNSFGLRF,0,1,0,0
5,Oxytocin,CYIQNCPLG,0,1,0,1
6,Gonadorelin,QHWSYGLRPG,0,1,1,0


## 3. Masses and m/z for 1+ to 4+

In [4]:
from peptide_mass import mass, mz
rows = []
for r in ex.itertuples():
    mono, avg = mass(r.sequence, n_acetyl=bool(r.n_acetyl), c_amide=bool(r.c_amide),
                     pyroglu=bool(r.pyroglu), disulfides=int(r.disulfides))
    rows.append({'name': r.name, 'mono': round(mono, 4), 'avg': round(avg, 2),
                 **{f'mz_{z}+': round(mz(mono, z), 4) for z in range(1, 5)}})
calc = pd.DataFrame(rows)
calc

,name,mono,avg,mz_1+,mz_2+,mz_3+,mz_4+
0,GHK,340.1859,340.38,341.1932,171.1002,114.4026,86.0538
1,Epitalon,390.1387,390.35,391.1460,196.0766,131.0535,98.5419
2,KPV,342.2267,342.43,343.2340,172.1206,115.0828,86.5640
3,DSIP,848.3301,848.81,849.3373,425.1723,283.7840,213.0898
4,Kisspeptin-10,1301.6305,1302.44,1302.6378,651.8225,434.8841,326.4149
5,Oxytocin,1006.4365,1007.19,1007.4437,504.2255,336.4861,252.6164
6,Gonadorelin,1181.5730,1182.29,1182.5803,591.7938,394.8650,296.4005


## 4. Compare with `data/masses.csv`
The check runs over every row; only the example rows are displayed.

In [5]:
allrows = []
for r in peps.itertuples():
    mono, avg = mass(r.sequence, bool(r.n_acetyl), bool(r.c_amide), bool(r.pyroglu), int(r.disulfides))
    allrows.append((r.name, mono, avg))
full = pd.DataFrame(allrows, columns=['name', 'mono', 'avg']).merge(ref, on='name')
full['d_mono_ppm'] = (full['mono'] - full['monoisotopic_da']) / full['monoisotopic_da'] * 1e6
full['d_avg'] = full['avg'] - full['average_da']
print('rows compared:', len(full))
print('max |delta mono| (ppm):', round(full['d_mono_ppm'].abs().max(), 4))
print('max |delta avg| (Da):', round(full['d_avg'].abs().max(), 4))
full[full['name'].isin(EXAMPLES)][['name', 'mono', 'monoisotopic_da', 'd_mono_ppm']].round(4)

rows compared: 23
max |delta mono| (ppm): 0.0551
max |delta avg| (Da): 0.0048


,name,mono,monoisotopic_da,d_mono_ppm
3,GHK,340.1859,340.1859,0.0118
4,Epitalon,390.1387,390.1387,-0.0538
6,KPV,342.2267,342.2267,0.0175
9,DSIP,848.3301,848.3301,-0.0448
11,Kisspeptin-10,1301.6305,1301.6305,0.0307
12,Oxytocin,1006.4365,1006.4365,-0.0407
13,Gonadorelin,1181.5730,1181.5730,0.0229


## 5. Modification example
The same nonapeptide CYIQNCPLG as a free acid, then C-terminally amidated, then with one disulfide, then also N-acetylated.

In [6]:
seq = 'CYIQNCPLG'
for label, kw in [('free acid', {}), ('+ C-amide', {'c_amide': True}),
                  ('+ C-amide + 1 S-S', {'c_amide': True, 'disulfides': 1}),
                  ('+ N-acetyl + C-amide + 1 S-S', {'n_acetyl': True, 'c_amide': True, 'disulfides': 1})]:
    mono, avg = mass(seq, **kw)
    print(f'{label:32s} mono {mono:10.4f}  avg {avg:10.2f}  [M+2H]2+ {mz(mono, 2):9.4f}')

free acid                        mono  1009.4361  avg    1010.19  [M+2H]2+  505.7253
+ C-amide                        mono  1008.4521  avg    1009.20  [M+2H]2+  505.2333
+ C-amide + 1 S-S                mono  1006.4365  avg    1007.19  [M+2H]2+  504.2255
+ N-acetyl + C-amide + 1 S-S     mono  1048.4470  avg    1049.22  [M+2H]2+  525.2308


For laboratory research use only. Not for human or veterinary use.